# RPO

- Arbitrary RAAN, inclination, and phasing
- Angles, then 3 DOF, then 6 DOF

In [46]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [47]:
import numpy as np
from numpy.linalg import norm
from cesium.constants import DEG_TO_RAD, KM_TO_M, M_TO_KM, RAD_TO_DEG, SEC_TO_MIN
from cesium.orbital.coes import coes_to_rv
from cesium.physics.dynamics import propagate_two_body
from cesium.plotting import plot_3d
from cesium.world.constants import MU_MARS, R_MARS

from cesium.guidance.coplanar import guid_coaxial_hohmann
from cesium.guidance.noncoplanar import guid_combined_i_raan_change

## Setup
Assume within 5 degrees inclination, raan, 0.05 e, 100km a

In [48]:
# Target orbit (same as arrival.ipynb), MCI, circular
PARKING_ORBIT_ALT_M = 300e3  # m
PARKING_ORBIT_RAAN_DEG = 15
PARKING_ORBIT_I_DEG = 45
PARKING_ORBIT_TA_DEG = 13

# a, e, i, raan, aop, ta [m, -, rad, rad, rad, rad]
TARGET_COES = np.array(
    [
        R_MARS + PARKING_ORBIT_ALT_M,
        0.0,
        PARKING_ORBIT_I_DEG * DEG_TO_RAD,
        PARKING_ORBIT_RAAN_DEG * DEG_TO_RAD,
        0.0,
        PARKING_ORBIT_TA_DEG * DEG_TO_RAD,
    ]
)

In [49]:
# Chaser = target + seeded uniform dispersion
SEED = 279
ANGLE_DISP_DEG = 3.0  # i, raan, aop
A_DISP_M = 5 * KM_TO_M
E_MAX = 0.01  # e >= 0, so draw from [0, E_MAX]
INITIAL_TA_OFFSET_DEG = 22

rng = np.random.default_rng(SEED)
rel_coes = np.array(
    [
        rng.uniform(-A_DISP_M, A_DISP_M),
        rng.uniform(0.0, E_MAX),
        *rng.uniform(-ANGLE_DISP_DEG, ANGLE_DISP_DEG, 3) * DEG_TO_RAD,
        INITIAL_TA_OFFSET_DEG * DEG_TO_RAD,  # ta: phasing left to the RPO problem
    ]
)
CHASER_COES = TARGET_COES + rel_coes

for name, coes in (("target", TARGET_COES), ("chaser", CHASER_COES)):
    a, e, i, raan, aop, ta = coes
    print(
        f"{name}: a = {a * M_TO_KM:.3f} km, e = {e:.5f}, i = {i * RAD_TO_DEG:.3f} deg, "
        f"raan = {raan * RAD_TO_DEG:.3f} deg, aop = {aop * RAD_TO_DEG:.3f} deg, ta = {ta * RAD_TO_DEG:.3f} deg"
    )

target: a = 3696.000 km, e = 0.00000, i = 45.000 deg, raan = 15.000 deg, aop = 0.000 deg, ta = 13.000 deg
chaser: a = 3691.505 km, e = 0.00854, i = 42.608 deg, raan = 13.063 deg, aop = -1.869 deg, ta = 35.000 deg


In [50]:
# Propagate both for a few target periods
N_ORBITS = 3
period_s = 2 * np.pi * np.sqrt(TARGET_COES[0] ** 3 / MU_MARS)
t_s = np.linspace(0, N_ORBITS * period_s, 3000)

rv_target_mci = propagate_two_body(
    np.concatenate(coes_to_rv(*TARGET_COES, MU_MARS)), t_s, MU_MARS
)
rv_chaser0 = np.concatenate(coes_to_rv(*CHASER_COES, MU_MARS))
rv_chaser_mci = propagate_two_body(
    rv_chaser0, t_s, MU_MARS
)

fig = plot_3d(
    [rv_target_mci[:, 0:3] * M_TO_KM, rv_chaser_mci[:, 0:3] * M_TO_KM],
    [t_s, t_s],
    labels=["Target", "Chaser"],
    trace_colors=["black", "royalblue"],
    bodies={"name": "Mars", "radius": R_MARS * M_TO_KM, "color": "orangered"},
    title=f"Target and chaser, {N_ORBITS} orbits (MCI)",
    units="km",
)

## Planning
- long way
  - raan and i
  - circularize
  - phasing
- short way
  - minimum combined plane change (but figure that out)

In [ ]:
# Estimation of dV cost
# raan and i
dv_i_raan = guid_combined_i_raan_change(
    CHASER_COES[2], TARGET_COES[2],
    CHASER_COES[3] - TARGET_COES[3],
    rv_chaser0[3:6],
    )
print(f"dv1 [m/s] = {dv_i_raan} ({norm(dv_i_raan):.3f})")

# a * (1 - e)
r_p_chaser = CHASER_COES[0] * (1 - CHASER_COES[1])
r_a_chaser = CHASER_COES[0] * (1 + CHASER_COES[1])
r_p_target = TARGET_COES[0] * (1 - TARGET_COES[1])
r_a_target = TARGET_COES[0] * (1 + TARGET_COES[1])
assert r_p_target == r_a_target

a_trans, t_trans, dv_a, dv_b = guid_coaxial_hohmann(r_p_chaser, r_a_chaser, r_p_target, r_a_target, mu=MU_MARS)
print(f"dv2 [m/s] = {dv_a} ({norm(dv_a):.3f})")
print(f"dv3 [m/s] = {dv_b} ({norm(dv_b):.3f})")
print(f"Transfer time: {t_trans * SEC_TO_MIN:.3f}m")



dv1 [m/s] = [-109.66818045   78.73058579   93.34276585] (164.129)
dv2 [m/s] = -6.2099446268666725 (6.210)
dv3 [m/s] = 8.344024843218904 (8.344)
Transfer time: 56.435m
